In [26]:
import pandas as pd
import numpy as np
import os, glob

if __name__ == "__main__":
    DATASET_PATH = "/media/research-student/KingstonSSD/FANET_Dataset/PM_Finetune/PM_RL_850ms/uav_scenario_0"
    SAVE_PATH = "/home/research-student/omnet-fanet/data-processing-scripts/journal_3_scripts/switch_time_pm_finetune/pm_rl850ms_uav_scenario_0_switch_time.csv"

    scenario_list = [f.path for f in os.scandir(DATASET_PATH) if f.is_dir()]
    results = []
    for scenario in scenario_list:
        scenario_name = scenario.split("/")[-1]
        runs = sorted(glob.glob("{}/Run-*_GCS-Tx.csv".format(scenario))) # Get the different runs for each scenario
        run_number = [run.split('/')[-1].split("_")[0].split("-")[-1] for run in runs]
        for run_num in run_number:
            switch_time_df = pd.read_csv("{}/Run-{}_Switch_Time.csv".format(scenario, run_num))
            if not switch_time_df.empty:
                switch_time = switch_time_df["Switch_Time"].values[0]
                results.append({"Scenario": scenario_name, "Run_Num": run_num, "Switch_Time": switch_time})

    results_df = pd.DataFrame(results)
    results_df.sort_values(by=["Scenario", "Run_Num"], inplace=True)
    results_df.to_csv(SAVE_PATH)

In [27]:
import pandas as pd
df_1 = pd.read_csv("switch_time/wp_uav_scenario_0_switch_time.csv", usecols=['Scenario', 'Run_Num', 'Switch_Time'])
df_2 = pd.read_csv("switch_time_pm_finetune/pm_rl850ms_uav_scenario_0_switch_time.csv", usecols=['Scenario', 'Run_Num', 'Switch_Time'])
df_1.sort_values(by=["Scenario", "Run_Num"], inplace=True)
df_2.sort_values(by=["Scenario", "Run_Num"], inplace=True)
df_1.reset_index(drop=True, inplace=True)
df_2.reset_index(drop=True, inplace=True)
df_1.equals(df_2)

False

In [4]:
import pandas as pd
import numpy as np

df1 = pd.read_csv("switch_time/wp_uav_scenario_0_switch_time.csv", index_col=False)
df2 = pd.read_csv("switch_time_pm_finetune/pm_gcs_rl500ms_uav_scenario_0_switch_time.csv",  index_col=False)

df1.reset_index(drop=True, inplace=True)
df2.reset_index(drop=True, inplace=True)

# df1.compare(df2)
df = pd.merge(df1, df2, on=['Scenario','Run_Num', 'Switch_Time'], how='outer', indicator='Exist')
diff = df.loc[df["Exist"]!="both"]
diff.to_csv("switch_time_pm_finetune/pm_gcs_rl500ms_uav_scenario_0_diff.csv")

In [ ]:
# Check if there is any "right_only"
diff.loc[df["Exist"]=="right_only"]

In [ ]:
# For Slurm
import pandas as pd
import numpy as np
import os, glob

if __name__ == "__main__":
    DATASET_PATH = "/home/rlim0005/FANET_Dataset/DJISpark_Obj3_Retransmission_Datasets/DV_Based_500_MANET_Interference"
    SAVE_PATH = "/home/rlim0005/FANET_Dataset/DJISpark_Obj3_Retransmission_Datasets/switch_time"
    SCENARIOS = ["manet_scenario_0", "manet_scenario_1", "manet_scenario_2"]
    SAVE_FILES = ["dv_500_manet_scenario_0_switch_time.csv", "dv_500_manet_scenario_1_switch_time.csv", "dv_500_manet_scenario_2_switch_time.csv"]

    for i in range(len(SCENARIOS)):
        scenario_list = [f.path for f in os.scandir(os.path.join(DATASET_PATH, SCENARIOS[i])) if f.is_dir()]
        results = []
        for scenario in scenario_list:
            scenario_name = scenario.split("/")[-1]
            runs = sorted(glob.glob("{}/Run-*_GCS-Tx.csv".format(scenario))) # Get the different runs for each scenario
            run_number = [run.split('/')[-1].split("_")[0].split("-")[-1] for run in runs]
            for run_num in run_number:
                switch_time_df = pd.read_csv("{}/Run-{}_Switch_Time.csv".format(scenario, run_num))
                if not switch_time_df.empty:
                    switch_time = switch_time_df["Switch_Time"].values[0]
                    results.append({"Scenario": scenario_name, "Run_Num": run_num, "Switch_Time": switch_time})

        results_df = pd.DataFrame(results)
        if not results_df.empty:
            results_df.sort_values(by=["Scenario", "Run_Num"], inplace=True)
            results_df.to_csv(os.path.join(SAVE_PATH, SAVE_FILES[i]))

In [3]:
# Date: 09/10/2024
# To combine the switch time for test cases with USI 100 ms and test cases with other USIs (66.7 ms, 20 ms, 10 ms)

import pandas as pd
import os
import glob

DIR1 = "/home/research-student/omnet-fanet/data-processing-scripts/journal_3_scripts/switch_time_usi100"
DIR2 = "/home/research-student/omnet-fanet/data-processing-scripts/journal_3_scripts/switch_time_usi_667_20_10"
SAVE_PATH = "/home/research-student/omnet-fanet/data-processing-scripts/journal_3_scripts/switch_time"

switch_files = [file for file in os.listdir(DIR1)  if os.path.isfile(os.path.join(DIR1, file))]
for file in switch_files:
    if os.path.isfile(os.path.join(DIR2,file)):
        df1 = pd.read_csv(os.path.join(DIR1, file))
        df2 = pd.read_csv(os.path.join(DIR2, file))
        df = pd.concat([df1, df2])
        df.to_csv(os.path.join(SAVE_PATH, file))

In [2]:
import pandas as pd

df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/test/UAVSpeed-8_BitRate-13_Height-135_Distance-270_Modulation-QPSK_UAVSendingInterval-20_MANETNumNode-10_MANETBitRate-6.5/Run-0_GCS-Rx.csv")
df["Delay"] = df['RxTime'] - df['TxTime'] # Calc delay of packets received
rel_df = df.loc[(df["Delay"] <= 0.04)].copy() # Get DF of reliable packets

rel_df.loc[rel_df["Packet_Name"]=="UAVData_3-26"]

,RxTime,TxTime,Packet_Name,Bytes,RSSI,SINR,Dest_Addr,Unnamed: 7,Delay
